# 16. Dual-Head Adaptive GNN with Residual Constraint R-GCN

This notebook implements a genuine **dual-head graph architecture** while preserving the successful decoder-conditioned adaptive selector.

```text
                              Knowledge graph
                                    │
                         frozen shared R-GCN trunk
                                    │
                    ┌───────────────┴────────────────┐
                    │                                │
             fusion states gᶠᵢ              constraint R-GCN head
                    │                         Δgᶜᵢ = R-GCNᶜ(gᶠ,E)
        BART graph cross-attention                    │
                    │                  zero-initialized residual adapter
              fused state hᶠₜ             gᶜᵢ = gᶠᵢ + γΔgᶜᵢ
                    │                                │
                    └───────────────┬────────────────┘
                                    │
                         adaptive selector MLP
                      [hᶠₜ ; gᶜᵢ ; coverageₜᵢ]
                                    │
                           NONE or entity i
                                    │
                       exact entity commitment
                                    │
                         optional Hard-v2 backstop
```

The new constraint branch is **function-preserving at initialization**: its output projection starts at zero, so before training the model is exactly equivalent to the existing epoch-5 adaptive selector.

## Table of contents

1. [Protocol](#1-protocol)  
2. [Environment](#2-environment)  
3. [Configuration](#3-configuration)  
4. [Load the frozen backbone](#4-load-the-frozen-backbone)  
5. [Metrics and Hard-v2 utilities](#5-metrics-and-hard-v2-utilities)  
6. [Selector supervision](#6-selector-supervision)  
7. [Dual-head architecture](#7-dual-head-architecture)  
8. [Load the current selector](#8-load-the-current-selector)  
9. [Development splits](#9-development-splits)  
10. [Teacher-forced features](#10-teacher-forced-features)  
11. [Intrinsic loss and evaluation](#11-intrinsic-loss-and-evaluation)  
12. [Initialization-equivalence test](#12-initialization-equivalence-test)  
13. [Training and early stopping](#13-training-and-early-stopping)  
14. [Restore the best checkpoint](#14-restore-the-best-checkpoint)  
15. [Adaptive decoding](#15-adaptive-decoding)  
16. [Development generation ablation](#16-development-generation-ablation)  
17. [Paired bootstrap diagnostics](#17-paired-bootstrap-diagnostics)  
18. [Optional repaired-test evaluation](#18-optional-repaired-test-evaluation)  
19. [Package outputs](#19-package-outputs)  
20. [Interpretation checklist](#20-interpretation-checklist)


## 1. Protocol

The principal comparison is:

```text
Current adaptive selector using shared fusion entity states
                            versus
Dual-head adaptive selector using specialized constraint entity states
```

Both arms use the same frozen BART/R-GCN fusion checkpoint, current selector initialization, entity-start labels, binary WebNLG coverage, exact entity commitment, `τ = 0.70`, compatibility margin `M = 8.0`, and optional Hard-v2.

The repaired test is disabled by default. Any later result on it must be described as **post-development descriptive evidence**, because the repaired WebNLG test distribution has already influenced the architecture.


## 2. Environment


In [ ]:
# 1. Bootstrap (Xet disabled BEFORE any HF import — lesson from 12b).
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip -q install "transformers>=4.44,<5" "huggingface_hub>=0.25,<1" nltk rouge-score accelerate sentencepiece sacremoses sacrebleu

import sys, json, pickle, random, time, shutil, subprocess, re, unicodedata, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
drive.mount('/content/drive')
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU runtime required'

In [ ]:
# 1.1 PyTorch Geometric.
try:
    import torch_geometric
    print('ok:', torch_geometric.__version__)
except Exception:
    tv = torch.__version__.split('+')[0]; cv = torch.version.cuda
    url = f'https://data.pyg.org/whl/torch-{tv}+cu{cv.replace(".", "")}.html'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', url])
    import torch_geometric

## 3. Configuration


In [ ]:
# Project paths.
PROJECT_DIR = '/content/drive/MyDrive/kg_llm_project'
PROCESSED_DIR = f'{PROJECT_DIR}/baseline-bart-webnlg/processed'
CKPT_FUSION = f'{PROJECT_DIR}/fusion_only_outputs/checkpoints/fusion_only/model_best.pt'
CURRENT_SELECTOR_DIR = f'{PROJECT_DIR}/selector_eval_v5_earlystop'
CURRENT_SELECTOR_CKPT = f'{CURRENT_SELECTOR_DIR}/selector_best_v5.pt'
CURRENT_SPLIT_MANIFEST = f'{CURRENT_SELECTOR_DIR}/selector_dev_splits_v5.json'
OUTPUT_DIR = f'{PROJECT_DIR}/dual_head_adaptive_eval_v1'
HV2_EVAL = f'{PROJECT_DIR}/hard_v2_eval_fixed_v4'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Reproducibility and sequence lengths.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda'
MAX_GEN_LEN, MAX_INPUT_LEN, MAX_TARGET_LEN = 128, 256, 128
LOCK_MIN_DEPTH, ESCAPE_MARGIN = 2, 10.0

# Dual-head architecture.
MODEL_DIM = 768
CONSTRAINT_BOTTLENECK = 256
CONSTRAINT_NUM_BASES = 8
CONSTRAINT_DROPOUT = 0.10

# Training.
MAX_DUAL_EPOCHS = 8
PATIENCE = 3
MIN_DELTA = 1e-4
TRAIN_BATCH_SIZE = 24
VAL_BATCH_SIZE = 16
HEAD_LR = 3e-4
SELECTOR_ENTITY_LR = 1e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NONE_WEIGHT = 0.30
DELTA_L2_WEIGHT = 1e-5
TRAIN_NONE_MLP = False

# Frozen decoding configuration from the current selector.
SELECTOR_TAU = 0.70
COMPATIBILITY_MARGIN = 8.0

# Evaluation policy.
RUN_DEV_GENERATION = True
RUN_REPAIRED_TEST = False
REGENERATE_MISSING_SHARED_TEST_ARMS = False
BOOTSTRAP_SAMPLES = 2000

for p in (f'{PROJECT_DIR}/fixed_ablation_common.py', PROCESSED_DIR, CKPT_FUSION, CURRENT_SELECTOR_CKPT):
    assert os.path.exists(p), f'missing: {p}'

shutil.copy(f'{PROJECT_DIR}/fixed_ablation_common.py', '/content/fixed_ablation_common.py')
if '/content' not in sys.path: sys.path.insert(0, '/content')
import fixed_ablation_common as fac
from fixed_ablation_common import (
    load_artifacts, FusionOnlyGNNModel, load_variant_checkpoint_resume,
    add_final_logits_bias, pad_kg_nodes, find_entity_token_spans,
)

from urllib.parse import quote
BART_LOCAL = '/content/bart-base-local'; os.makedirs(BART_LOCAL, exist_ok=True)
FILES = {
    'config.json': 1000,
    'vocab.json': 800000,
    'merges.txt': 400000,
    'tokenizer.json': 1000000,
    'model.safetensors': 500000000,
}
for fn, mn in FILES.items():
    dest = os.path.join(BART_LOCAL, fn)
    if os.path.isfile(dest) and os.path.getsize(dest) >= mn:
        continue
    url = f'https://huggingface.co/facebook/bart-base/resolve/main/{quote(fn)}?download=true'
    rc = subprocess.run([
        'curl','-L','--fail','--retry','12','--retry-delay','5','--retry-all-errors',
        '--connect-timeout','30','--speed-time','90','--speed-limit','1024',
        '-C','-','-o',dest + '.part',url,
    ]).returncode
    if rc != 0:
        if os.path.exists(dest + '.part'): os.remove(dest + '.part')
        rc = subprocess.run(['curl','-L','--fail','-o',dest + '.part',url]).returncode
    assert rc == 0 and os.path.getsize(dest + '.part') >= mn, f'download failed: {fn}'
    os.replace(dest + '.part', dest)
    print('downloaded', fn)
print('BART snapshot ready')
print('Output directory:', OUTPUT_DIR)


## 4. Load the frozen backbone


In [ ]:
# 3. Load data + frozen fusion model.
from transformers import BartTokenizer, BartForConditionalGeneration
tokenizer = BartTokenizer.from_pretrained(BART_LOCAL)
bart = BartForConditionalGeneration.from_pretrained(BART_LOCAL, local_files_only=True).to(DEVICE)
data, graphs, vocab = load_artifacts(PROCESSED_DIR)
num_relations = len(vocab['relation_vocab']) * 2
model = FusionOnlyGNNModel(bart, num_relations=num_relations).to(DEVICE)
model = load_variant_checkpoint_resume(model, CKPT_FUSION, DEVICE, strict=False)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print('frozen fusion model ready | splits:', {k: len(v) for k, v in data.items()})

In [ ]:
fusion_model = model
fusion_model.eval()
for p in fusion_model.parameters(): p.requires_grad_(False)
print('Frozen fusion backbone registered as fusion_model.')


## 5. Metrics and Hard-v2 utilities


In [ ]:
# 4. Manifest + metrics + TrieMapV2 + hard mask (ports of the 12b-verified cells).
from collections import OrderedDict
from difflib import SequenceMatcher
import sacrebleu

def _tp(t):
    if isinstance(t, dict): return str(t.get('subject','')), str(t.get('predicate','')), str(t.get('object',''))
    return str(t[0]), str(t[1]), str(t[2])

groups = OrderedDict()
for i, ex in enumerate(data['test']):
    key = json.dumps([_tp(t) for t in ex['triples']], ensure_ascii=False)
    g = groups.setdefault(key, {'first_index': i, 'references': []})
    for v in list(ex.get('all_targets') or []) + [ex.get('target','')]:
        v = str(v).strip()
        if v and v not in g['references']: g['references'].append(v)
train_predicates = {_tp(t)[1] for ex in data['train'] for t in ex['triples']}
ITEMS = []
for uid, (key, g) in enumerate(groups.items()):
    ex = dict(data['test'][g['first_index']]); ex['all_targets'] = g['references']
    ITEMS.append({'uid': uid, 'idx': g['first_index'], 'ex': ex,
                  'unseen': bool({_tp(t)[1] for t in ex['triples']} - train_predicates)})
assert len(ITEMS) == 2510 and sum(x['unseen'] for x in ITEMS) == 752

_TR = {'ø':'o','Ø':'o','æ':'ae','Æ':'ae','œ':'oe','Œ':'oe','ð':'d','Ð':'d','þ':'th','Þ':'th',
       'ł':'l','Ł':'l','ß':'ss','đ':'d','Đ':'d','ħ':'h','ı':'i','İ':'i','ŋ':'ng'}
_MONTHS = ['january','february','march','april','may','june','july','august','september','october','november','december']
_DET = re.compile(r'^(the|a|an)\s+')
_STOP = frozenset(('the a an and or but if then this that these those it its he she they them his her their '
                   'in on at of to by with for from as is are was were be been being there here also however').split())
def _sa(s):
    s = ''.join(_TR.get(c, c) for c in str(s))
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
def _norm(s):
    s = _sa(str(s)).lower().strip().strip('"').strip("'")
    s = re.sub(r'[^a-z0-9 ]', ' ', s); s = re.sub(r'\s+', ' ', s).strip()
    return _DET.sub('', s)
def _wm(t, s):
    return bool(s) and re.search(r'(?<![a-z0-9])' + re.escape(s) + r'(?![a-z0-9])', t) is not None
def _dst(v):
    toks = set(); raw = _sa(str(v)).strip().strip('"').strip("'")
    m = re.match(r'^(\d{3,4})-(\d{1,2})-(\d{1,2})$', raw)
    if m:
        y, mo, d = map(int, m.groups()); toks.add(str(y))
        if 1 <= mo <= 12: toks.add(_MONTHS[mo-1]); toks.add(_MONTHS[mo-1][:3])
        toks.update({str(d), str(d).zfill(2)}); toks.update({f'{d}{sx}' for sx in ('st','nd','rd','th')})
    elif re.match(r'^\d{3,4}$', raw): toks.add(raw)
    return toks
def _dg(s): return re.sub(r'[^0-9]', '', str(s))
def grounding_score(prediction, triples):
    pn = _norm(prediction)
    forms, tokens, num = [], set(), set()
    for t in triples:
        s, p, o = _tp(t)
        for v in (s, o):
            n = _norm(v)
            if n: forms.append(n); tokens.update(n.split())
            tokens.update(_dst(v)); d = _dg(v)
            if d: num.add(d)
        tokens.update(_norm(re.sub(r'([a-z])([A-Z])', r'\1 \2', p)).split())
    fd = [f.replace(' ', '') for f in forms]
    found = sum(1 for f in forms if _wm(pn, f) or all(_wm(pn, w) for w in f.split()))
    recall = found / len(forms) if forms else 1.0
    men = set()
    for m in re.findall(r'[A-Z][A-Za-z]*(?:[ -][A-Z][A-Za-z]*)*', _sa(prediction)):
        n = _norm(m)
        if len(n) >= 3 and (' ' in n or n not in _STOP): men.add(n)
    for m in re.findall(r'[A-Za-z0-9]+(?:[./\-][A-Za-z0-9]+)*', str(prediction)):
        if any(c.isdigit() for c in m): men.add(_norm(m))
    men = {m for m in men if m}
    def ok(m):
        for f in forms:
            if m == f or _wm(f, m): return True
        if all(t in tokens for t in m.split()): return True
        d = _dg(m)
        if d and any(d in c or c in d for c in num): return True
        md = m.replace(' ', '')
        return len(md) >= 3 and any(md in x or x in md for x in fd)
    hall = sorted(m for m in men if not ok(m))
    corr = [m for m in hall if max((SequenceMatcher(None, m, f).ratio() for f in forms), default=0) >= 0.55]
    return {'halluc': len(hall)/len(men) if men else 0.0, 'recall': recall,
            'hallucinated': hall, 'corruptions': corr}
ART = {'iso': r'[A-Za-z],? \d{3,4}-\d{1,2}-\d{1,2}|\d{1,2}(st|nd|rd|th)? [A-Za-z]+ \d{3,4}-\d{1,2}-\d{1,2}',
       'paren': r'\((The [^)]+album|[0-9]{4} film|film|band|song|album|actor[^)]*|footballer[^)]*|musician[^)]*)\)',
       'unit': r'\d[\d.,]*\s*\((milli|centi|kilo)?(metres|meters|grams|litres|liters|inches)\)'}
def artrow(p): return any(re.search(x, str(p)) for x in ART.values())
def corpus_bleu_lc(preds, refs_list):
    maxr = max(len(r) for r in refs_list)
    streams = [[r[k] if k < len(r) else r[0] for r in refs_list] for k in range(maxr)]
    return sacrebleu.corpus_bleu(preds, streams, lowercase=True, tokenize='13a').score

class TNode:
    __slots__ = ('ch', 'score', 'mx', 'nterm', 'terminal')
    def __init__(self):
        self.ch = {}; self.score = None; self.mx = -1e9; self.nterm = 0; self.terminal = False
_LIT = [r'^[\d\s.,:/\-+%°"]*$', r'^\d{3,4}-\d{1,2}-\d{1,2}', r'^\d+(\.\d+)?$']
def is_literal(name):
    n = str(name).strip().strip('"').strip("'").strip()
    return len(n) < 2 or any(re.match(p, n) for p in _LIT)
def clean_surface(name):
    s = str(name).strip().strip('"').strip("'").replace('_', ' ')
    s = re.sub(r'\s*\([^)]*\)', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
class TrieMapV2:
    def __init__(self, entity_names, tokenizer):
        self.root = TNode(); self.kept = {}
        for ni, name in enumerate(entity_names):
            if is_literal(name): continue
            base = clean_surface(name)
            if not base or is_literal(base): continue
            self.kept[ni] = base
            for v in (base, ' ' + base):
                ids = tokenizer.encode(v, add_special_tokens=False)
                if not ids: continue
                n = self.root
                for tid in ids: n = n.ch.setdefault(int(tid), TNode())
                n.terminal = True
        self._cache(self.root)
    def _cache(self, n):
        nt = 1 if n.terminal else 0
        for c in n.ch.values():
            self._cache(c); nt += c.nterm
        n.nterm = nt
    def suffix_matches(self, gen_ids, lookback=20):
        out = []
        for start in range(max(0, len(gen_ids) - lookback), len(gen_ids)):
            n = self.root; okk = True
            for pos in range(start, len(gen_ids)):
                t = int(gen_ids[pos])
                if t not in n.ch: okk = False; break
                n = n.ch[t]
            if okk and n is not self.root: out.append((len(gen_ids) - start, n))
        return out
def hard_mask_v2(logits, trie, gen_ids):
    best = None
    for depth, node in trie.suffix_matches(gen_ids):
        if node.terminal or not node.ch: continue
        if (depth >= LOCK_MIN_DEPTH or node.nterm == 1) and (best is None or depth > best[0]):
            best = (depth, node)
    if best is None: return logits, False
    legal = list(best[1].ch.keys())
    if float(logits.max()) - max(float(logits[t]) for t in legal) > ESCAPE_MARGIN:
        return logits, False
    m = torch.full_like(logits, -1e9); m[legal] = 0.0
    return logits + m, True
print('protocol cells ready')

## 6. Selector supervision


In [ ]:
# 5. Selector module + gold span labels from references.
class EntitySelector(nn.Module):
    # scores {NONE} ∪ {entities} from [h_t ; e_i ; cov_i]
    def __init__(self, d=768, hid=256):
        super().__init__()
        self.ent_mlp = nn.Sequential(nn.Linear(2*d + 1, hid), nn.ReLU(), nn.Linear(hid, 1))
        self.none_mlp = nn.Sequential(nn.Linear(d, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, h, ents, cov):
        # h: (L,d) | ents: (N,d) | cov: (L,N) in {0,1}
        L, d = h.shape; N = ents.shape[0]
        he = torch.cat([h.unsqueeze(1).expand(L, N, d), ents.unsqueeze(0).expand(L, N, d),
                        cov.unsqueeze(-1)], dim=-1)
        s_ent = self.ent_mlp(he).squeeze(-1)          # (L,N)
        s_none = self.none_mlp(h)                     # (L,1)
        return torch.cat([s_none, s_ent], dim=-1)     # (L, 1+N); class 0 = NONE

def target_labels(ex, graph, target_ids):
    # label per decoder-state position t (predicting token t of target_ids):
    #  0 = NONE, i+1 = entity i STARTS at t, -100 = inside a span (ignored)
    names = list(getattr(graph, 'entity_names', []) or [])
    surfaces = [clean_surface(n) if not is_literal(n) else '' for n in names]
    lab = np.zeros(len(target_ids), dtype=np.int64)
    text = ex['target']
    spans = find_entity_token_spans(text, [s if s else '§none§' for s in surfaces], tokenizer)
    for ni, (s, e) in enumerate(spans):
        if s < 0 or not surfaces[ni]: continue
        if s < len(lab): lab[s] = ni + 1
        for k in range(s + 1, min(e, len(lab))): lab[k] = -100
    return lab
print('selector defined')

## 7. Dual-head architecture


In [ ]:
from torch_geometric.nn import RGCNConv

class ResidualConstraintRGCNHead(nn.Module):
    """One relation-aware constraint head with function-preserving initialization."""
    def __init__(self, d=768, bottleneck=256, num_relations=None, num_bases=8, dropout=0.1):
        super().__init__()
        assert num_relations is not None
        self.conv = RGCNConv(d, bottleneck, num_relations, num_bases=min(num_bases, num_relations))
        self.norm = nn.LayerNorm(bottleneck)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(bottleneck, d)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
        self.residual_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, g_fusion, edge_index, edge_type):
        h = self.conv(g_fusion, edge_index, edge_type)
        h = self.dropout(self.norm(F.gelu(h)))
        delta = self.out(h)
        g_constraint = g_fusion + self.residual_scale * delta
        return g_constraint, delta

class DualHeadAdaptiveGNN(nn.Module):
    def __init__(self, d, bottleneck, num_relations, num_bases, dropout):
        super().__init__()
        self.constraint_head = ResidualConstraintRGCNHead(
            d=d, bottleneck=bottleneck, num_relations=num_relations,
            num_bases=num_bases, dropout=dropout,
        )
        self.selector = EntitySelector(d=d, hid=256)

    def constraint_states(self, g_fusion, edge_index, edge_type):
        return self.constraint_head(g_fusion, edge_index, edge_type)

def nparams(module, trainable=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable))

print('Dual-head architecture ready.')


## 8. Load the current selector


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def atomic_json_dump(value, path, indent=2):
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(value, f, indent=indent, ensure_ascii=False)
    os.replace(tmp, path)

def atomic_torch_save(value, path):
    tmp = path + '.tmp'
    torch.save(value, tmp)
    os.replace(tmp, path)

current_ckpt = torch.load(CURRENT_SELECTOR_CKPT, map_location=DEVICE)
current_selector = EntitySelector(d=MODEL_DIM, hid=256).to(DEVICE)
current_selector.load_state_dict(current_ckpt['state'])
current_selector.eval()
for p in current_selector.parameters(): p.requires_grad_(False)

dual_model = DualHeadAdaptiveGNN(
    d=MODEL_DIM,
    bottleneck=CONSTRAINT_BOTTLENECK,
    num_relations=num_relations,
    num_bases=CONSTRAINT_NUM_BASES,
    dropout=CONSTRAINT_DROPOUT,
).to(DEVICE)
dual_model.selector.load_state_dict(current_ckpt['state'])

# Freeze everything first, then explicitly enable the new branch and entity MLP.
for p in dual_model.parameters(): p.requires_grad_(False)
for p in dual_model.constraint_head.parameters(): p.requires_grad_(True)
for p in dual_model.selector.ent_mlp.parameters(): p.requires_grad_(True)
for p in dual_model.selector.none_mlp.parameters(): p.requires_grad_(TRAIN_NONE_MLP)

print('Current selector epoch:', current_ckpt.get('epoch'))
print('Current selector SHA256:', sha256_file(CURRENT_SELECTOR_CKPT))
print('Constraint-head parameters:', f'{nparams(dual_model.constraint_head):,}')
print('Trainable dual-head parameters:', f'{nparams(dual_model, trainable=True):,}')
print('Trainable names:')
for name, p in dual_model.named_parameters():
    if p.requires_grad: print(' ', name)


## 9. Development splits


In [ ]:
def rebuild_v5_splits():
    universe = list(range(len(data['dev'])))
    assert len(universe) == 1667
    a = set(random.Random(7).sample(universe, 300))
    pool = [i for i in universe if i not in a]
    b = set(random.Random(21).sample(pool, 500))
    pool = [i for i in universe if i not in a and i not in b]
    c = set(random.Random(33).sample(pool, 300))
    unused = [i for i in universe if i not in a and i not in b and i not in c]
    val = set(random.Random(55).sample(unused, 167))
    remaining = [i for i in unused if i not in val]
    tune = set(random.Random(66).sample(remaining, 200))
    confirm = set(remaining) - tune
    groups = {
        'excluded_historical_seed7': sorted(a),
        'excluded_historical_seed21': sorted(b),
        'excluded_inspected_v4_tune': sorted(c),
        'selector_validation': sorted(val),
        'generation_tune': sorted(tune),
        'generation_confirm': sorted(confirm),
    }
    raw = json.dumps(groups, sort_keys=True, separators=(',', ':')).encode()
    return {'version': 'v5_compatible_dual_head', 'dev_size': 1667, 'groups': groups,
            'sha256': hashlib.sha256(raw).hexdigest()}

if os.path.exists(CURRENT_SPLIT_MANIFEST):
    SPLITS = json.load(open(CURRENT_SPLIT_MANIFEST, encoding='utf-8'))
else:
    SPLITS = rebuild_v5_splits()
SELECTOR_VAL_IDX = SPLITS['groups']['selector_validation']
GEN_TUNE_IDX = SPLITS['groups']['generation_tune']
GEN_CONFIRM_IDX = SPLITS['groups']['generation_confirm']
assert len(SELECTOR_VAL_IDX) == 167 and len(GEN_TUNE_IDX) == 200 and len(GEN_CONFIRM_IDX) == 200
assert not set(SELECTOR_VAL_IDX) & set(GEN_TUNE_IDX)
assert not set(SELECTOR_VAL_IDX) & set(GEN_CONFIRM_IDX)
assert not set(GEN_TUNE_IDX) & set(GEN_CONFIRM_IDX)
atomic_json_dump(SPLITS, os.path.join(OUTPUT_DIR, 'dual_head_split_manifest.json'))
print({k: len(SPLITS['groups'][k]) for k in ('selector_validation','generation_tune','generation_confirm')})


## 10. Teacher-forced features


In [ ]:
from torch_geometric.data import Batch as PyGBatch
pad_id = tokenizer.pad_token_id
dec_start = fusion_model.bart.config.decoder_start_token_id

@torch.no_grad()
def teacher_features(examples, graph_list):
    enc_batch = tokenizer(
        [x['linearized'] for x in examples], max_length=MAX_INPUT_LEN,
        truncation=True, padding=True, return_tensors='pt',
    ).to(DEVICE)
    tgt_batch = tokenizer(
        text_target=[x['target'] for x in examples], max_length=MAX_TARGET_LEN,
        truncation=True, padding=True, return_tensors='pt',
    )
    target_ids = tgt_batch['input_ids']
    decoder_ids = torch.cat([
        torch.full((target_ids.size(0), 1), dec_start, dtype=target_ids.dtype),
        target_ids[:, :-1],
    ], dim=1).to(DEVICE)
    gb = PyGBatch.from_data_list(graph_list).to(DEVICE)
    enc = fusion_model.bart.model.encoder(
        input_ids=enc_batch['input_ids'], attention_mask=enc_batch['attention_mask'])
    g_fusion, _ = fusion_model.rgcn(gb.x, gb.edge_index, gb.edge_type)
    g_pad, g_mask = pad_kg_nodes(g_fusion, gb.batch, len(examples))
    dec = fusion_model.bart.model.decoder(
        input_ids=decoder_ids, encoder_hidden_states=enc.last_hidden_state,
        encoder_attention_mask=enc_batch['attention_mask'])
    h_fused, _, _ = fusion_model.kg_cross_attention(dec.last_hidden_state, g_pad, g_mask)
    return {
        'h_fused': h_fused.detach(), 'g_fusion': g_fusion.detach(),
        'batch': gb.batch.detach(), 'edge_index': gb.edge_index.detach(),
        'edge_type': gb.edge_type.detach(), 'target_ids': target_ids,
    }

def coverage_from_labels(labels, n_entities):
    cov = torch.zeros(len(labels), n_entities, device=DEVICE)
    seen = set()
    for t in range(len(labels)):
        for i in seen: cov[t, i] = 1.0
        y = int(labels[t].item())
        if y > 0: seen.add(y - 1)
    return cov

def example_pack(ex, graph, target_row, h_row, entity_states):
    names = list(getattr(graph, 'entity_names', []) or [])
    if not names: return None
    assert entity_states.size(0) == len(names), (entity_states.size(0), len(names))
    L = int((target_row != pad_id).sum().item())
    if L < 2: return None
    labels = torch.from_numpy(target_labels(ex, graph, target_row[:L].tolist())).long().to(DEVICE)
    return {
        'h': h_row[:L].float(), 'ents': entity_states.float(),
        'labels': labels, 'cov': coverage_from_labels(labels, len(names)), 'L': L,
    }


## 11. Intrinsic loss and evaluation


In [ ]:
def selector_ce(logits, labels):
    weights = torch.ones(logits.size(-1), device=logits.device)
    weights[0] = NONE_WEIGHT
    return F.cross_entropy(logits, labels, weight=weights, ignore_index=-100)

@torch.no_grad()
def evaluate_intrinsic(mode, indices, batch_size=VAL_BATCH_SIZE, verbose=True):
    assert mode in ('shared', 'dual')
    fusion_model.eval(); current_selector.eval(); dual_model.eval()
    tp = fp = fn = exact = true_starts = none_ok = none_n = 0
    loss_sum = 0.0; loss_n = 0; delta_sum = 0.0; delta_n = 0
    for start in range(0, len(indices), batch_size):
        ids = indices[start:start+batch_size]
        examples = [data['dev'][i] for i in ids]
        glist = [graphs['dev'][i] for i in ids]
        ft = teacher_features(examples, glist)
        if mode == 'dual':
            all_ents, delta = dual_model.constraint_states(
                ft['g_fusion'].float(), ft['edge_index'], ft['edge_type'])
            delta_sum += float(delta.norm(dim=-1).sum().cpu()); delta_n += delta.size(0)
        else:
            all_ents = ft['g_fusion'].float()
        for j, (ex, graph) in enumerate(zip(examples, glist)):
            node_idx = (ft['batch'] == j).nonzero(as_tuple=False).flatten()
            pack = example_pack(ex, graph, ft['target_ids'][j], ft['h_fused'][j], all_ents[node_idx])
            if pack is None: continue
            selector_obj = current_selector if mode == 'shared' else dual_model.selector
            logits = selector_obj(pack['h'], pack['ents'], pack['cov'])
            loss = selector_ce(logits, pack['labels'])
            if torch.isfinite(loss): loss_sum += float(loss.cpu()); loss_n += 1
            pred = logits.argmax(-1).cpu(); gold = pack['labels'].cpu()
            for t in range(pack['L']):
                y = int(gold[t]); p = int(pred[t])
                if y == -100: continue
                if y > 0:
                    true_starts += 1
                    if p > 0:
                        tp += 1
                        if p == y: exact += 1
                    else: fn += 1
                else:
                    none_n += 1
                    if p > 0: fp += 1
                    else: none_ok += 1
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    F1 = 2 * P * R / max(P + R, 1e-12)
    out = {
        'mode': mode, 'n_examples': len(indices), 'loss': loss_sum/max(loss_n,1),
        'P': P, 'R': R, 'F1': F1,
        'identity_accuracy_among_detected_starts': exact/max(tp,1),
        'NONE_accuracy': none_ok/max(none_n,1), 'true_starts': true_starts,
        'detected_gold_starts': tp, 'false_triggers': fp, 'missed_starts': fn,
        'exact_entity_identity': exact, 'none_positions': none_n,
        'mean_delta_norm': delta_sum/max(delta_n,1) if mode == 'dual' else 0.0,
        'residual_scale': float(dual_model.constraint_head.residual_scale.detach().cpu()) if mode == 'dual' else 0.0,
    }
    if verbose:
        print(f"{mode:>6} | loss={out['loss']:.4f} P={P:.4f} R={R:.4f} F1={F1:.4f} "
              f"identity={out['identity_accuracy_among_detected_starts']:.4f} "
              f"NONE={out['NONE_accuracy']:.4f} delta={out['mean_delta_norm']:.4f}")
    return out


## 12. Initialization-equivalence test


In [ ]:
dual_model.eval(); current_selector.eval()
probe_examples = [data['train'][0], data['train'][1]]
probe_graphs = [graphs['train'][0], graphs['train'][1]]
ft = teacher_features(probe_examples, probe_graphs)
with torch.no_grad():
    g_c, delta = dual_model.constraint_states(ft['g_fusion'].float(), ft['edge_index'], ft['edge_type'])
max_rep_diff = float((g_c - ft['g_fusion'].float()).abs().max().cpu())
max_delta = float(delta.abs().max().cpu())
logit_diffs = []
for j, (ex, graph) in enumerate(zip(probe_examples, probe_graphs)):
    idx = (ft['batch'] == j).nonzero(as_tuple=False).flatten()
    pack = example_pack(ex, graph, ft['target_ids'][j], ft['h_fused'][j], ft['g_fusion'][idx])
    if pack is None: continue
    with torch.no_grad():
        a = current_selector(pack['h'], pack['ents'], pack['cov'])
        b = dual_model.selector(pack['h'], g_c[idx].float(), pack['cov'])
    logit_diffs.append(float((a-b).abs().max().cpu()))
max_logit_diff = max(logit_diffs, default=0.0)
print({'max_delta': max_delta, 'max_representation_difference': max_rep_diff,
       'max_selector_logit_difference': max_logit_diff})
assert max_delta < 1e-8 and max_rep_diff < 1e-8 and max_logit_diff < 1e-7
print('PASS: epoch-0 dual-head behavior exactly matches the current selector.')


## 13. Training and early stopping

Only the new constraint R-GCN head and the selector's entity MLP are trainable. The `NONE` MLP remains frozen. The objective is weighted multiclass cross-entropy plus a small residual-drift penalty:

\[
\mathcal{L}=\mathcal{L}_{CE}+\lambda_\Deltarac{1}{N}\sum_i\|\Delta g_i^C\|_2^2.
\]

Epoch 0 is saved as a valid baseline checkpoint. A trained checkpoint replaces it only if validation improves without a meaningful `NONE`-accuracy collapse.


In [ ]:
LAST_PATH = os.path.join(OUTPUT_DIR, 'dual_head_last.pt')
BEST_PATH = os.path.join(OUTPUT_DIR, 'dual_head_best.pt')
HISTORY_PATH = os.path.join(OUTPUT_DIR, 'dual_head_train_history.json')
BASELINE_PATH = os.path.join(OUTPUT_DIR, 'current_selector_intrinsic_baseline.json')

if os.path.exists(BASELINE_PATH):
    shared_base = json.load(open(BASELINE_PATH, encoding='utf-8'))
else:
    shared_base = evaluate_intrinsic('shared', SELECTOR_VAL_IDX)
    atomic_json_dump(shared_base, BASELINE_PATH)

optimizer = torch.optim.AdamW([
    {'params': [p for p in dual_model.constraint_head.parameters() if p.requires_grad],
     'lr': HEAD_LR, 'weight_decay': WEIGHT_DECAY},
    {'params': [p for p in dual_model.selector.ent_mlp.parameters() if p.requires_grad],
     'lr': SELECTOR_ENTITY_LR, 'weight_decay': WEIGHT_DECAY},
])

def better(candidate, best, baseline):
    if candidate['NONE_accuracy'] < baseline['NONE_accuracy'] - 0.005: return False
    if candidate['F1'] > best['F1'] + MIN_DELTA: return True
    tied = abs(candidate['F1'] - best['F1']) <= MIN_DELTA
    if tied and candidate['identity_accuracy_among_detected_starts'] > best['identity_accuracy_among_detected_starts'] + MIN_DELTA:
        return True
    return False

if not os.path.exists(BEST_PATH):
    init_metrics = evaluate_intrinsic('dual', SELECTOR_VAL_IDX)
    atomic_torch_save({'state': dual_model.state_dict(), 'epoch': 0,
                       'validation': init_metrics, 'baseline_validation': shared_base}, BEST_PATH)
    print('Saved epoch-0 function-preserving checkpoint.')

best_obj = torch.load(BEST_PATH, map_location=DEVICE)
best_metrics = dict(best_obj['validation']); best_epoch = int(best_obj['epoch'])
start_epoch = 0; history = []; no_improve = 0
if os.path.exists(LAST_PATH):
    resume = torch.load(LAST_PATH, map_location=DEVICE)
    dual_model.load_state_dict(resume['state']); optimizer.load_state_dict(resume['optimizer'])
    start_epoch = int(resume['epoch']); history = list(resume.get('history', []))
    best_metrics = dict(resume.get('best_metrics', best_metrics)); best_epoch = int(resume.get('best_epoch', best_epoch))
    no_improve = int(resume.get('epochs_without_improvement', 0))
    print('Resumed at epoch', start_epoch, '| best', best_epoch)

train_indices = list(range(len(data['train'])))
for epoch in range(start_epoch, MAX_DUAL_EPOCHS):
    dual_model.train(); fusion_model.eval(); random.Random(SEED + epoch).shuffle(train_indices)
    ce_sum = reg_sum = total_sum = 0.0; updates = 0; t0 = time.time()
    for start in range(0, len(train_indices), TRAIN_BATCH_SIZE):
        ids = train_indices[start:start+TRAIN_BATCH_SIZE]
        examples = [data['train'][i] for i in ids]
        glist = [graphs['train'][i] for i in ids]
        ft = teacher_features(examples, glist)
        g_c, delta = dual_model.constraint_states(ft['g_fusion'].float(), ft['edge_index'], ft['edge_type'])
        ce = None; usable = 0
        for j, (ex, graph) in enumerate(zip(examples, glist)):
            node_idx = (ft['batch'] == j).nonzero(as_tuple=False).flatten()
            pack = example_pack(ex, graph, ft['target_ids'][j], ft['h_fused'][j], g_c[node_idx])
            if pack is None: continue
            loss_i = selector_ce(dual_model.selector(pack['h'], pack['ents'], pack['cov']), pack['labels'])
            if torch.isfinite(loss_i):
                ce = loss_i if ce is None else ce + loss_i; usable += 1
        if usable == 0: continue
        ce = ce / usable; reg = delta.pow(2).mean(); total = ce + DELTA_L2_WEIGHT * reg
        optimizer.zero_grad(set_to_none=True); total.backward()
        torch.nn.utils.clip_grad_norm_([p for p in dual_model.parameters() if p.requires_grad], GRAD_CLIP)
        optimizer.step()
        ce_sum += float(ce.detach().cpu()); reg_sum += float(reg.detach().cpu()); total_sum += float(total.detach().cpu()); updates += 1
        if updates % 50 == 0:
            print(f'epoch {epoch+1} step {updates}: CE={ce_sum/updates:.4f} total={total_sum/updates:.4f} '
                  f'scale={float(dual_model.constraint_head.residual_scale.detach().cpu()):.4f} ({time.time()-t0:.0f}s)')
    val = evaluate_intrinsic('dual', SELECTOR_VAL_IDX)
    improved = better(val, best_metrics, shared_base)
    if improved:
        best_metrics = dict(val); best_epoch = epoch + 1; no_improve = 0
        atomic_torch_save({'state': dual_model.state_dict(), 'epoch': best_epoch,
                           'validation': val, 'baseline_validation': shared_base,
                           'config': {'head_lr': HEAD_LR, 'selector_entity_lr': SELECTOR_ENTITY_LR,
                                      'none_weight': NONE_WEIGHT, 'delta_l2_weight': DELTA_L2_WEIGHT}}, BEST_PATH)
        print('NEW BEST', best_epoch, best_metrics['F1'], best_metrics['identity_accuracy_among_detected_starts'])
    else:
        no_improve += 1; print('No safe improvement:', no_improve, '/', PATIENCE)
    row = {'epoch': epoch+1, 'mean_ce_loss': ce_sum/max(updates,1),
           'mean_regularization': reg_sum/max(updates,1), 'mean_total_loss': total_sum/max(updates,1),
           'updates': updates, 'elapsed_sec': time.time()-t0, 'validation': val,
           'is_best': improved, 'best_epoch_after_this_epoch': best_epoch,
           'epochs_without_improvement': no_improve}
    history.append(row); atomic_json_dump(history, HISTORY_PATH)
    stop = no_improve >= PATIENCE
    atomic_torch_save({'state': dual_model.state_dict(), 'optimizer': optimizer.state_dict(),
                       'epoch': epoch+1, 'history': history, 'best_metrics': best_metrics,
                       'best_epoch': best_epoch, 'epochs_without_improvement': no_improve,
                       'early_stopped': stop}, LAST_PATH)
    if stop:
        print('Early stopping.'); break


## 14. Restore the best checkpoint


In [ ]:
best_ckpt = torch.load(BEST_PATH, map_location=DEVICE)
dual_model.load_state_dict(best_ckpt['state']); dual_model.eval()
for p in dual_model.parameters(): p.requires_grad_(False)
BEST_EPOCH = int(best_ckpt['epoch']); BEST_SHA = sha256_file(BEST_PATH)
shared_val = evaluate_intrinsic('shared', SELECTOR_VAL_IDX)
dual_val = evaluate_intrinsic('dual', SELECTOR_VAL_IDX)
summary = {
    'best_epoch': BEST_EPOCH, 'checkpoint': BEST_PATH, 'sha256': BEST_SHA,
    'current_selector_sha256': sha256_file(CURRENT_SELECTOR_CKPT),
    'shared_validation': shared_val, 'dual_validation': dual_val,
    'delta': {'F1': dual_val['F1']-shared_val['F1'],
              'identity': dual_val['identity_accuracy_among_detected_starts']-shared_val['identity_accuracy_among_detected_starts'],
              'NONE': dual_val['NONE_accuracy']-shared_val['NONE_accuracy']},
}
atomic_json_dump(summary, os.path.join(OUTPUT_DIR, 'dual_head_intrinsic_summary.json'))
import pandas as pd
from IPython.display import display
display(pd.DataFrame([shared_val, dual_val]))
print('Best epoch:', BEST_EPOCH, '| SHA256:', BEST_SHA)
print('Residual scale:', float(dual_model.constraint_head.residual_scale.detach().cpu()))
if os.path.exists(HISTORY_PATH):
    hist = json.load(open(HISTORY_PATH, encoding='utf-8'))
    display(pd.DataFrame([{'epoch': x['epoch'], 'train_ce': x['mean_ce_loss'],
                           'val_F1': x['validation']['F1'],
                           'val_identity': x['validation']['identity_accuracy_among_detected_starts'],
                           'val_NONE': x['validation']['NONE_accuracy'],
                           'delta_norm': x['validation']['mean_delta_norm'], 'best': x['is_best']} for x in hist]))


## 15. Adaptive decoding


In [ ]:
@torch.no_grad()
def decode_adaptive(ex, graph, mode='dual', use_hard=True, tau=SELECTOR_TAU, margin=COMPATIBILITY_MARGIN, return_events=False):
    assert mode in ('shared', 'dual')
    enc_in = tokenizer(ex['linearized'], max_length=MAX_INPUT_LEN, truncation=True,
                       padding='max_length', return_tensors='pt')
    inp, att = enc_in['input_ids'].to(DEVICE), enc_in['attention_mask'].to(DEVICE)
    gb = torch.zeros(graph.x.size(0), dtype=torch.long, device=DEVICE)
    enc = fusion_model.bart.model.encoder(input_ids=inp, attention_mask=att)
    edge_index, edge_type = graph.edge_index.to(DEVICE), graph.edge_type.to(DEVICE)
    g_fusion, _ = fusion_model.rgcn(graph.x.to(DEVICE), edge_index, edge_type)
    h_pad, k_mask = pad_kg_nodes(g_fusion, gb, 1)
    if mode == 'dual':
        selector_ents, _ = dual_model.constraint_states(g_fusion.float(), edge_index, edge_type)
        selector_obj = dual_model.selector
    else:
        selector_ents = g_fusion.float(); selector_obj = current_selector
    names = list(getattr(graph, 'entity_names', []) or [])
    trie = TrieMapV2(names, tokenizer)
    ent_ids = {}
    for ni, base in trie.kept.items():
        a = tokenizer.encode(base, add_special_tokens=False)
        b = tokenizer.encode(' ' + base, add_special_tokens=False)
        if a and b: ent_ids[ni] = (a, b)
    start_id, eos = fusion_model.bart.config.decoder_start_token_id, fusion_model.bart.config.eos_token_id
    gen = [start_id]; past = None; committed = None; selector_triggers = 0; hard_triggers = 0; events = []
    for step in range(MAX_GEN_LEN - 1):
        di = torch.tensor([[gen[-1]]], dtype=torch.long, device=DEVICE)
        out = fusion_model.bart.model.decoder(
            input_ids=di, encoder_hidden_states=enc.last_hidden_state,
            encoder_attention_mask=att, past_key_values=past, use_cache=True)
        past = out.past_key_values
        h, _, _ = fusion_model.kg_cross_attention(out.last_hidden_state, h_pad, k_mask)
        logits = add_final_logits_bias(fusion_model.bart, fusion_model.bart.lm_head(h))[:, -1, :].squeeze(0).float().cpu()
        if committed is not None:
            nxt = committed['ids'][committed['pos']]; committed['pos'] += 1
            if committed['pos'] >= len(committed['ids']): committed = None
        else:
            if use_hard:
                logits2, active = hard_mask_v2(logits, trie, gen[1:]); hard_triggers += int(active)
            else: logits2 = logits
            nxt = int(logits2.argmax())
            if ent_ids:
                decoded = tokenizer.decode(gen[1:], skip_special_tokens=True); dn = _norm(decoded)
                cov_flags = [1.0 if (trie.kept.get(i) and _wm(dn, _norm(trie.kept[i]))) else 0.0
                             for i in range(len(names))]
                cov = torch.tensor([cov_flags], device=DEVICE)
                sel = selector_obj(h[:, -1, :].float(), selector_ents.float(), cov).squeeze(0)
                ent_logits = sel[1:].clone()
                for i in range(len(names)):
                    if i not in ent_ids or cov_flags[i] > 0: ent_logits[i] = -1e9
                probs = torch.softmax(torch.cat([sel[:1], ent_logits]), -1)
                if probs.numel() > 1:
                    best_i = int(probs[1:].argmax()); p_best = float(probs[1+best_i])
                    if p_best >= tau and best_i in ent_ids:
                        seq = ent_ids[best_i][0] if len(gen) == 1 else ent_ids[best_i][1]
                        gap = float(logits.max()) - float(logits[seq[0]])
                        if gap <= margin:
                            nxt = seq[0]; selector_triggers += 1
                            if len(seq) > 1: committed = {'ids': seq, 'pos': 1}
                            events.append({'step': step, 'entity_index': best_i,
                                           'entity': trie.kept.get(best_i, names[best_i]),
                                           'probability': p_best, 'bart_first_token_gap': gap})
        if nxt == eos: break
        gen.append(nxt)
    result = {'prediction': tokenizer.decode(gen[1:], skip_special_tokens=True),
              'selector_triggers': selector_triggers, 'hard_v2_triggers': hard_triggers}
    if return_events: result['events'] = events
    return result

print('Decoder ready. TrieMap is used only by Hard-v2; selector commitment uses the chosen entity token sequence directly.')


## 16. Development generation ablation


In [ ]:
def dev_items(indices):
    out = []
    for i in indices:
        ex = dict(data['dev'][i])
        refs = ex.get('all_targets') or [ex.get('target','')]
        ex['all_targets'] = [str(x).strip() for x in refs if str(x).strip()]
        out.append({'idx': i, 'ex': ex, 'graph': graphs['dev'][i]})
    return out

def run_cached(items, tag, mode, use_hard):
    cache_tag = f'{tag}_{BEST_SHA[:12]}'
    ppath = os.path.join(OUTPUT_DIR, f'dev_preds_{cache_tag}.json')
    tpath = os.path.join(OUTPUT_DIR, f'dev_triggers_{cache_tag}.json')
    preds = json.load(open(ppath)) if os.path.exists(ppath) else []
    trigs = json.load(open(tpath)) if os.path.exists(tpath) else []
    assert len(preds) == len(trigs) <= len(items)
    t0 = time.time()
    for i in range(len(preds), len(items)):
        r = decode_adaptive(items[i]['ex'], items[i]['graph'], mode=mode, use_hard=use_hard)
        preds.append(r['prediction']); trigs.append({'selector': r['selector_triggers'], 'hard_v2': r['hard_v2_triggers']})
        if (i+1) % 25 == 0 or i+1 == len(items):
            atomic_json_dump(preds, ppath); atomic_json_dump(trigs, tpath)
        if (i+1) % 100 == 0: print(f'[{tag}] {i+1}/{len(items)} ({time.time()-t0:.0f}s)')
    return preds, trigs

def score_block(preds, items):
    gs = [grounding_score(p, item['ex']['triples']) for p, item in zip(preds, items)]
    return {'n': len(items),
            'bleu': corpus_bleu_lc(preds, [item['ex']['all_targets'] for item in items]),
            'halluc': float(np.mean([g['halluc'] for g in gs])),
            'recall': float(np.mean([g['recall'] for g in gs])),
            'corr_rows': int(sum(len(g['corruptions']) > 0 for g in gs)),
            'art_rows': int(sum(artrow(p) for p in preds))}

DEV_RESULTS, DEV_PREDS = {}, {}
if RUN_DEV_GENERATION:
    split_map = {'tune': GEN_TUNE_IDX, 'confirm': GEN_CONFIRM_IDX}
    arms = {
        'shared_selector': ('shared', False),
        'dual_head_selector': ('dual', False),
        'shared_selector_hard_v2': ('shared', True),
        'dual_head_selector_hard_v2': ('dual', True),
    }
    for split, indices in split_map.items():
        items = dev_items(indices); DEV_RESULTS[split] = {}; DEV_PREDS[split] = {}
        for arm, (mode, hard) in arms.items():
            preds, _ = run_cached(items, f'{split}_{arm}', mode, hard)
            DEV_PREDS[split][arm] = preds; DEV_RESULTS[split][arm] = score_block(preds, items)
    frame = pd.DataFrame([{'split': s, 'arm': a, **m} for s,d in DEV_RESULTS.items() for a,m in d.items()])
    display(frame); frame.to_csv(os.path.join(OUTPUT_DIR, 'dual_head_dev_summary.csv'), index=False)
    atomic_json_dump(DEV_RESULTS, os.path.join(OUTPUT_DIR, 'dual_head_dev_summary.json'))
    for split in ('tune','confirm'):
        for suffix in ('','_hard_v2'):
            a = DEV_RESULTS[split]['shared_selector'+suffix]; b = DEV_RESULTS[split]['dual_head_selector'+suffix]
            print(split, suffix or '_no_hard', {'BLEU': b['bleu']-a['bleu'],
                  'recall_pp': 100*(b['recall']-a['recall']),
                  'halluc_pp': 100*(b['halluc']-a['halluc']),
                  'corr_rows': b['corr_rows']-a['corr_rows']})
else:
    print('Development generation disabled.')


## 17. Paired bootstrap diagnostics


In [ ]:
def paired_bootstrap(dual_preds, shared_preds, items, key, samples=BOOTSTRAP_SAMPLES, seed=20260725):
    d = np.array([grounding_score(a, it['ex']['triples'])[key] - grounding_score(b, it['ex']['triples'])[key]
                  for a,b,it in zip(dual_preds, shared_preds, items)], dtype=float)
    rng = np.random.default_rng(seed)
    boot = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(samples)])
    if key == 'recall': improved, worsened = int((d>0).sum()), int((d<0).sum())
    else: improved, worsened = int((d<0).sum()), int((d>0).sum())
    return {'delta': float(d.mean()), 'ci95': [float(np.percentile(boot,2.5)), float(np.percentile(boot,97.5))],
            'improved': improved, 'worsened': worsened, 'tied': int((d==0).sum())}

PAIRED = {}
if RUN_DEV_GENERATION and DEV_PREDS:
    for split, indices in {'tune': GEN_TUNE_IDX, 'confirm': GEN_CONFIRM_IDX}.items():
        items = dev_items(indices); PAIRED[split] = {}
        for name, dual_arm, shared_arm in [
            ('dual_vs_shared','dual_head_selector','shared_selector'),
            ('dual_hard_vs_shared_hard','dual_head_selector_hard_v2','shared_selector_hard_v2')]:
            PAIRED[split][name] = {k: paired_bootstrap(DEV_PREDS[split][dual_arm], DEV_PREDS[split][shared_arm], items, k)
                                    for k in ('recall','halluc')}
            PAIRED[split][name]['changed_outputs'] = int(sum(a != b for a,b in zip(DEV_PREDS[split][dual_arm], DEV_PREDS[split][shared_arm])))
    atomic_json_dump(PAIRED, os.path.join(OUTPUT_DIR, 'dual_head_dev_paired.json'))
    print(json.dumps(PAIRED, indent=2))
else:
    print('No development predictions available.')


## 18. Optional repaired-test evaluation

`RUN_REPAIRED_TEST` is `False` by default. Enable it only after all architecture and training choices are frozen. The notebook reuses the previously generated current-selector arms when available and decodes only the two new dual-head arms.


In [ ]:
def find_file(name):
    for root, _, files in os.walk(PROJECT_DIR):
        if name in files: return os.path.join(root, name)
    return None

def run_test_arm(tag, mode, hard):
    cache_tag = f'{tag}_{BEST_SHA[:12]}'
    ppath = os.path.join(OUTPUT_DIR, f'test_preds_{cache_tag}.json')
    tpath = os.path.join(OUTPUT_DIR, f'test_triggers_{cache_tag}.json')
    preds = json.load(open(ppath)) if os.path.exists(ppath) else []
    trigs = json.load(open(tpath)) if os.path.exists(tpath) else []
    assert len(preds) == len(trigs) <= len(ITEMS)
    t0 = time.time()
    for i in range(len(preds), len(ITEMS)):
        item = ITEMS[i]
        r = decode_adaptive(item['ex'], graphs['test'][item['idx']], mode=mode, use_hard=hard)
        preds.append(r['prediction']); trigs.append({'selector': r['selector_triggers'], 'hard_v2': r['hard_v2_triggers']})
        if (i+1) % 25 == 0 or i+1 == len(ITEMS):
            atomic_json_dump(preds, ppath); atomic_json_dump(trigs, tpath)
        if (i+1) % 250 == 0: print(f'[{tag}] {i+1}/2510 ({time.time()-t0:.0f}s)')
    return preds

def test_block(preds, indices):
    P = [preds[i] for i in indices]; its = [ITEMS[i] for i in indices]
    gs = [grounding_score(p, it['ex']['triples']) for p,it in zip(P,its)]
    return {'n': len(indices), 'bleu': corpus_bleu_lc(P, [it['ex']['all_targets'] for it in its]),
            'halluc': float(np.mean([g['halluc'] for g in gs])),
            'recall': float(np.mean([g['recall'] for g in gs])),
            'corr_rows': int(sum(len(g['corruptions']) > 0 for g in gs)),
            'art_rows': int(sum(artrow(p) for p in P))}

if RUN_REPAIRED_TEST:
    shared_path = find_file('preds_fusion_selector_only.json')
    shared_hard_path = find_file('preds_fusion_selector_hard_v2.json')
    if shared_path and shared_hard_path:
        shared = json.load(open(shared_path)); shared_hard = json.load(open(shared_hard_path))
    elif REGENERATE_MISSING_SHARED_TEST_ARMS:
        shared = run_test_arm('shared_selector','shared',False)
        shared_hard = run_test_arm('shared_selector_hard_v2','shared',True)
    else:
        raise FileNotFoundError('Previous shared-selector test predictions not found. Do not silently regenerate them.')
    dual = run_test_arm('dual_head_selector','dual',False)
    dual_hard = run_test_arm('dual_head_selector_hard_v2','dual',True)
    arms = {'shared_selector': shared, 'dual_head_selector': dual,
            'shared_selector_hard_v2': shared_hard, 'dual_head_selector_hard_v2': dual_hard}
    subsets = {'overall': list(range(len(ITEMS))),
               'seen': [i for i,x in enumerate(ITEMS) if not x['unseen']],
               'unseen': [i for i,x in enumerate(ITEMS) if x['unseen']]}
    TEST_SUMMARY = {'status': 'post-development descriptive; not pristine confirmatory',
                    'checkpoint': {'path': BEST_PATH, 'sha256': BEST_SHA, 'epoch': BEST_EPOCH},
                    'frozen_decode': {'tau': SELECTOR_TAU, 'margin': COMPATIBILITY_MARGIN},
                    'arms': {a: {s: test_block(p,idx) for s,idx in subsets.items()} for a,p in arms.items()}}
    atomic_json_dump(TEST_SUMMARY, os.path.join(OUTPUT_DIR, 'dual_head_test_descriptive.json'))
    display(pd.DataFrame([{'arm':a,'subset':s,**m} for a,d in TEST_SUMMARY['arms'].items() for s,m in d.items()]))
else:
    print('Repaired-test decoding disabled — intended default.')


## 19. Package outputs


In [ ]:
manifest = {
    'architecture': 'frozen shared R-GCN/BART fusion + residual constraint R-GCN head + adaptive selector + optional Hard-v2',
    'fusion_checkpoint': CKPT_FUSION,
    'current_selector_checkpoint': CURRENT_SELECTOR_CKPT,
    'dual_best_checkpoint': BEST_PATH if os.path.exists(BEST_PATH) else None,
    'dual_best_sha256': sha256_file(BEST_PATH) if os.path.exists(BEST_PATH) else None,
    'config': {
        'seed': SEED, 'bottleneck': CONSTRAINT_BOTTLENECK, 'num_bases': CONSTRAINT_NUM_BASES,
        'dropout': CONSTRAINT_DROPOUT, 'max_epochs': MAX_DUAL_EPOCHS, 'patience': PATIENCE,
        'head_lr': HEAD_LR, 'selector_entity_lr': SELECTOR_ENTITY_LR,
        'none_weight': NONE_WEIGHT, 'delta_l2_weight': DELTA_L2_WEIGHT,
        'tau': SELECTOR_TAU, 'compatibility_margin': COMPATIBILITY_MARGIN,
        'run_repaired_test': RUN_REPAIRED_TEST,
    },
    'scientific_status': 'development experiment; any repaired-test result is descriptive',
}
atomic_json_dump(manifest, os.path.join(OUTPUT_DIR, 'run_manifest.json'))
zip_path = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print('Archive:', zip_path)
for f in sorted(os.listdir(OUTPUT_DIR)): print(' ', f)


## 20. Interpretation checklist

A larger model is not automatically a stronger contribution. The new branch is useful only if it improves over the existing adaptive selector under the same decoding configuration.

| Signal | Desired result |
|---|---|
| Entity-start F1 | Higher or unchanged |
| Entity identity accuracy | Higher |
| `NONE` accuracy | No meaningful collapse |
| Entity recall | Higher |
| Entity hallucination | Lower or unchanged |
| BLEU | No meaningful degradation |
| Corruption rows | Lower or unchanged |
| Seen/unseen robustness | Preferably improved |

### Possible outcomes

- **Best epoch > 0 and generation improves:** specialized relation-aware constraint representations add value beyond the shared fusion embedding.
- **Epoch 0 remains best:** the current selector already extracts sufficient information from the fusion representation; do not claim a dual-head gain.
- **Intrinsic gains but no generation gains:** the remaining problem is likely autoregressive exposure, coverage, thresholding, or exact-commitment policy rather than graph representation capacity.
- **Gains only with Hard-v2:** learned entity initiation and deterministic entity completion remain complementary.

After a positive result, the architecture can be described as:

> A dual-head relational graph encoder in which a shared frozen graph representation supports language-model fusion, while a residual relation-aware constraint head specializes entity representations for decoder-conditioned selection.
